# Baba Is You — RLVR with GRPO (Colab Demo)

This notebook trains an LLM agent on the *Baba Is You* OpenEnv RLVR environment using **Group Relative Policy Optimization** (GRPO) via Hugging Face TRL + Unsloth.

Pipeline:
1. Clone repo, install deps.
2. Launch the OpenEnv FastAPI server in the background.
3. Generate a small MAP-Elites curriculum.
4. Run a baseline (random agent) — record success rate.
5. Run GRPO training on Qwen2.5-3B-Instruct (4-bit LoRA).
6. Re-evaluate trained agent — record success rate.
7. Plot reward curve from W&B.

In [ ]:
!git clone https://github.com/YOUR_USER/baba-rlvr.git
%cd baba-rlvr
!pip install -q uv && uv sync --extra train --extra train-unsloth

In [ ]:
# Launch server in the background
import subprocess, time, requests
proc = subprocess.Popen(['uv', 'run', 'baba-server'])
for _ in range(20):
    try:
        if requests.get('http://localhost:8000/health').ok:
            break
    except Exception:
        time.sleep(0.5)
print(requests.get('http://localhost:8000/levels').json())

In [ ]:
# Generate a small curriculum (fast)
!uv run baba-pcg generate --iterations 500 --max-depth 12 --out levels/archive.pkl

In [ ]:
# Baseline: random agent on the tutorial
!uv run baba-eval random --episodes 30 --level-id tutorial_01

In [ ]:
# GRPO training (~20 min on a T4)
import os
os.environ['WANDB_PROJECT'] = 'baba-rlvr'
!uv run python -m baba_rlvr.training.grpo_train \
    --env-url http://localhost:8000 \
    --model unsloth/Qwen2.5-3B-Instruct-bnb-4bit \
    --steps 100 \
    --num-generations 4 \
    --max-turns 30 \
    --use-memory false

## Ablation: with vs without memory scratchpad
Re-run training with `--use-memory true` and compare success rates on a held-out set of PCG levels.